In [1]:
from planktonzilla.dataset import *

/lustre/fswork/projects/rech/tec/uod68bo/planktonzilla/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset, concatenate_datasets
from tqdm import tqdm
import numpy as np
import torch

In [3]:
ds_ood = load_dataset("project-oceania/planktonzilla_ood")
test_id = load_dataset("project-oceania/planktonzilla_only_plankton", split="test")

In [4]:
train_ood, val_ood, test_ood = gen_splits(ds_ood, 0.2, 0.2, 42)

Generating new stratified splits.


In [5]:
ds_ood = concatenate_datasets([train_ood, val_ood, test_ood])

In [6]:
ood_names = ds_ood.features["label"].names
ood_names_dataset = np.array(ood_names)[ds_ood["label"]]

In [7]:
id_names = test_id.features["label"].names
id_names_dataset = np.array(id_names)[test_id["label"]]

In [8]:
labels_final = np.concatenate([ood_names_dataset, id_names_dataset])

In [9]:
ood_datasets = np.array(ds_ood["dataset"])
id_datasets = np.array(test_id["dataset"])

datasets_final = np.concatenate([ood_datasets, id_datasets])

In [10]:
mahalanobis_scores = torch.load("/lustre/fswork/projects/rech/tec/uod68bo/planktonzilla/logs/train_ood/runs/2026-02-25_01-58-10_369386/mahalanobis_FINAL.pt")
vim_scores = torch.load("/lustre/fswork/projects/rech/tec/uod68bo/planktonzilla/logs/train_ood/runs/2026-02-25_01-58-10_369386/vim_FINAL.pt")

In [17]:
mahalanobis_scores

{'scores': tensor([0.0002, 0.0003, 0.0003,  ..., 0.0001, 0.0001, 0.0001]),
 'y': tensor([-1, -1, -1,  ..., 79, 94, 41])}

In [18]:
from datasets import Dataset
import numpy as np

data = {
    "vim_score": vim_scores["scores"].numpy(),
    "mahalanobis_score": mahalanobis_scores["scores"].numpy(),
    "label": labels_final,        # strings
    "dataset": datasets_final,    # strings
    "class": vim_scores["y"].numpy()
}


In [19]:
from datasets import Features, Value

features = Features({
    "vim_score": Value("float32"),
    "mahalanobis_score": Value("float32"),
    "label": Value("string"),
    "dataset": Value("string"),
    "class": Value("int64"),
})

hf_ds = Dataset.from_dict(data, features=features)

In [20]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo(
    repo_id="project-oceania/ood_eva02_clf_scores",
    repo_type="dataset",
    private=False,
    exist_ok=True
)

RepoUrl('https://huggingface.co/datasets/project-oceania/ood_eva02_clf_scores', endpoint='https://huggingface.co', repo_type='dataset', repo_id='project-oceania/ood_eva02_clf_scores')

In [21]:
hf_ds.push_to_hub("project-oceania/EVA02_CLF_OOD_scores")

Creating parquet from Arrow format: 100%|██████████| 4/4 [00:01<00:00,  2.38ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  11%|█▏        | 8.85MB / 78.2MB, 7.38MB/s  
Processing Files (0 / 1):  13%|█▎        | 10.0MB / 78.2MB, 7.16MB/s  
Processing Files (0 / 1):  20%|█▉        | 15.3MB / 78.2MB, 9.60MB/s  
Processing Files (0 / 1):  35%|███▍      | 27.2MB / 78.2MB, 15.1MB/s  
Processing Files (0 / 1):  41%|████      | 32.0MB / 78.2MB, 16.0MB/s  
Processing Files (0 / 1):  54%|█████▍    | 42.1MB / 78.2MB, 19.2MB/s  
Processing Files (0 / 1):  62%|██████▏   | 48.8MB / 78.2MB, 20.3MB/s  
Processing Files (0 / 1):  73%|███████▎  | 57.2MB / 78.2MB, 22.0MB/s  
Processing Files (0 / 1):  85%|████████▌ | 66.7MB / 78.2MB, 23.8MB/s  
Processing Files (0 / 1):  93%|█████████▎| 72.7MB / 78.2MB, 24.2MB/s  
Processing Files (0 / 1): 100%|█████████▉| 78.1MB / 78.2MB, 24.4MB/s  
Processing Files (1 / 1): 100%|██████████| 78.2MB / 78.2MB, 18.6MB/s  

CommitInfo(commit_url='https://huggingface.co/datasets/project-oceania/EVA02_CLF_OOD_scores/commit/f11c81e23b24c4aa67281cee443e1983cfd3f5ba', commit_message='Upload dataset', commit_description='', oid='f11c81e23b24c4aa67281cee443e1983cfd3f5ba', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/project-oceania/EVA02_CLF_OOD_scores', endpoint='https://huggingface.co', repo_type='dataset', repo_id='project-oceania/EVA02_CLF_OOD_scores'), pr_revision=None, pr_num=None)